## Carregamendo do gabarito  
  

Com base no conteúdo rotulado nas duas bases BOS.xlsx e DECC.xlsx aqui é realizado o carregamento do gabarito.

In [1]:
import os
import re
import unicodedata
import pandas as pd
from unidecode import unidecode

In [2]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\joaol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
PATH_BASE_BOS = os.getenv('PATH_BASE_BOS')
PATH_BASE_DECCC = os.getenv('PATH_BASE_DECCC')
PATH_GABARITO = os.getenv('PATH_GABARITO')

### Carregando a Base

In [4]:
df_origin = pd.read_excel(PATH_BASE_BOS, usecols=['n_bop', 'relato', 'NOMES']).dropna(subset=['NOMES']).reset_index()
df_new = pd.read_excel(PATH_BASE_DECCC, usecols=['n_bop', 'relato', 'nomes']).dropna(subset=['nomes']).reset_index()

df = pd.DataFrame({'n_bop':pd.concat([df_origin['n_bop'], df_new['n_bop']], ignore_index=True),
                    'relato':pd.concat([df_origin['relato'], df_new['relato']], ignore_index=True),
                    'nomes': pd.concat([df_origin['NOMES'], df_new['nomes']], ignore_index=True)})
df

,n_bop,relato,nomes
0,00615/2021.100191-1,"A DECLARANTE ACIMA QUALIFICADA, INFORMA QUE E...","MANDA, FARIAS, SERIA, POLO, WILLIAN"
1,00616/2021.100053-9,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...,"EDUARDO, HELIO, PEREIRA, DIAS, MALA, ELIAS, IR..."
2,00616/2021.100190-3,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...,"EDUARDO, SILVA, PEREIRA, JOSE, PAULO, MARTINIA..."
3,00616/2021.100222-0,A RELATORA ACIMA QUALIFICADA COMPARECE NESTA ...,ERALDO
4,00616/2021.100016-0,O RELATOR ACIMA QUALIFICADO COMPARECE NESTA D...,"FERNANDA, VIVIANE, IRIA, ALEM, REGINA"
...,...,...,...
766,00487/2020.100303-9,O RELATOR COMPARECE NESTA DIVISAO DE POLICIA ...,"ELITO, FONSECA, CORREA"
767,00615/2020.100363-4,A VITIMA RELATA QUE NO DIA E HORA SUPRACITADO...,"WELLINGTON, ALAN, MACEDO, CHAVES, LEONARDO, AR..."
768,00615/2020.100847-3,A CIDADA ACIMA QUALIFICADA COMPARECEU NESTA D...,"MARCIA, PRISCILA, MARIA, EDILVA, ALVES, SILVA"
769,00615/2020.100746-2,A RELATORA COMPARECE NESTA DIVISAO DE POLICIA...,"JACKELINE, GELENE,"


In [3]:
df = pd.read_csv(r'C:\Users\joaol\OneDrive\Desktop\MestradoBaseMédica\ProjetoDoutorado\BASES\base_dados_medica_com_id.csv', usecols=['relato', 'nomes']).dropna(subset=['nomes']).reset_index(drop=True)

### Pré-processamento nos relatos

In [4]:
def preprocess(relato):
    def remove_stop_words(relato):
        # Remove stop words em português
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        # Remoção de números e pontuação    
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()

In [5]:
df['relato'] = df['relato'].apply(normalize_text)
df['relato_tratado'] = df['relato'].apply(preprocess)

### Extração dos nomes da coluna de rótulos

In [6]:
coluna_nomes_base = [re.split(r', | |,', df['nomes'][i]) for i in range(0,len(df))]

In [7]:
def load_names_base(coluna_nomes_base):
    lista_nomes = []
    for value in coluna_nomes_base:
        for non in value:
            lista_nomes.append(non)
    print("Tamanho original antes da remoção de duplicados: ", len(lista_nomes))
    return set(lista_nomes)

In [8]:
nomes = load_names_base(coluna_nomes_base)

Tamanho original antes da remoção de duplicados:  13400


In [9]:
print("Tamanho após a remoção dos duplicados: ", len(nomes))

Tamanho após a remoção dos duplicados:  638


### Obtenção da coluna de relatos e transformar em tokens

In [10]:
coluna_relato_base = [df['relato_tratado'][i].split(' ') for i in range(0,len(df))]

In [11]:
def load_words_base(coluna_relato_base):
    lista_relatos = []
    for value in coluna_relato_base:
        for word in value:
            lista_relatos.append(word)
    print("Tamanho original antes da remoção de duplicados: ", len(lista_relatos))
    return set(lista_relatos)

In [12]:
palavras = load_words_base(coluna_relato_base)

Tamanho original antes da remoção de duplicados:  231798


In [13]:
print("Tamanho após a remoção dos duplicados: ", len(palavras))

Tamanho após a remoção dos duplicados:  11528


### Veficação da existencia de dados repetidos

In [14]:
df_nomes = pd.DataFrame({'nomes':list(nomes)})
df_palavras = pd.DataFrame({'palavras':list(palavras)})

### Removendo valores vazios nas celulas

In [15]:
df_nomes = df_nomes.replace({'':None})
df_nomes = df_nomes.dropna()

In [16]:
df_palavras = df_palavras.replace({'':None})
df_palavras = df_palavras.dropna()

In [17]:
df_palavras

,palavras
0,VASOPRESSORA
1,HIDRICA
2,OSTEOPOROSE
3,FAMILIA
4,NOTED
...,...
11523,ICARAI
11524,PROCEDIMENTO
11525,AMARO
11526,PONTUACAO


In [18]:
df_palavras[df_palavras['palavras'].isin(df_nomes['nomes'].to_list())]

,palavras
5,MAURICIO
34,PORTELA
87,BORGES
101,JULIO
104,SIQUEIRA
...,...
11472,NOBREGA
11475,SANTOS
11483,LUCIANO
11492,SALGADO


In [19]:
df_palavras = df_palavras[~df_palavras['palavras'].isin(df_nomes['nomes'].to_list())].reset_index(drop=True)

In [20]:
df_palavras[df_palavras['palavras'].isin(df_nomes['nomes'].to_list())]

,palavras


In [21]:
df_palavras

,palavras
0,VASOPRESSORA
1,HIDRICA
2,OSTEOPOROSE
3,FAMILIA
4,NOTED
...,...
10988,ICARAI
10989,PROCEDIMENTO
10990,AMARO
10991,PONTUACAO


In [22]:
df_nomes['y'] = 1
df_palavras['y'] = 0

In [23]:
tokens = df_nomes['nomes'].to_list() + df_palavras['palavras'].to_list()

In [24]:
tokens

['MARIETA',
 'LUZ',
 'ELIANE',
 'GUSMÃO',
 'MAURICIO',
 'CARLOS',
 'SÔNIA',
 'ABREU',
 'DIEGO',
 'MARTHA',
 'PIETRA',
 'BRAGA',
 'EVANDRO',
 'ULISSES',
 'CLÁUDIA',
 'ROMILDO',
 'LUAN',
 'JUAREZ',
 'MANUELA',
 'TEREZA',
 'PORTELA',
 'FERRAZ',
 'JOCIELE',
 'LÚCIA',
 'VINICIUS',
 'VINÍCIUS',
 'ASSIS',
 'PEREIRA',
 'HEITOR',
 'ESTEVÃO',
 'CECÍLIA',
 'MADUREIRA',
 'MAISONNAVE',
 'MARCUS',
 'JOÃO',
 'FUJITA',
 'LEONARDO',
 'CAMILA',
 'E',
 'RIOLO',
 'CARINA',
 'YARA',
 'MARISTELA',
 'MASCARENHAS',
 'MOLINAS',
 'MENEZES',
 'ALFREDO',
 'JONAS',
 'PÊGO',
 'LOUREIRO',
 'FRAGA',
 'VERDE',
 'BICALHO',
 'EDGAR',
 'ANSELMO',
 'SOUZA',
 'CRISTINA',
 'GUERRA',
 'LEANDRO',
 'FREITAS',
 'SIMÕES',
 'ROGÉRIO',
 'THAIS',
 'MARTINS',
 'RITA',
 'F',
 'MIRANDA',
 'JÉSSICA',
 'MOURA',
 'HUGO',
 'ESPINOSA',
 'MARQUEZ',
 'BIA',
 'ORLANDO',
 'BORGES',
 'LÍDIA',
 'MARCOS',
 'NOVAES',
 'SENNA',
 'TEREZINHA',
 'CECCON',
 'ELENA',
 'MATOS',
 'MUNIZ',
 'PRADO',
 'MURILO',
 'JULIO',
 'CASSIA',
 'C.',
 'SIQUEIRA',
 'AGU

In [25]:
rotulos = df_nomes['y'].to_list() + df_palavras['y'].to_list()

In [26]:
gabarito = pd.DataFrame({'palavra':tokens, 'y':rotulos}).dropna()

In [27]:
gabarito

,palavra,y
0,MARIETA,1
1,LUZ,1
2,ELIANE,1
3,GUSMÃO,1
4,MAURICIO,1
...,...,...
11626,ICARAI,0
11627,PROCEDIMENTO,0
11628,AMARO,0
11629,PONTUACAO,0


In [28]:
gabarito = gabarito.drop_duplicates(subset=['palavra'])
gabarito

,palavra,y
0,MARIETA,1
1,LUZ,1
2,ELIANE,1
3,GUSMÃO,1
4,MAURICIO,1
...,...,...
11626,ICARAI,0
11627,PROCEDIMENTO,0
11628,AMARO,0
11629,PONTUACAO,0


### Correção gab - removendo stopwords que sobraram

Alguns nomes possuem tokens "stopwords" como "AQUELES" ou "AQUELE" essa etapa é pra realizar a remoçao destes. Devido problemas que ocorreram no momento da avalização, os tokenizadores do Bert e Spacy removeram porém passou no nosso, por isso estes casos foram removidos.

In [29]:
stopwords_list = stopwords.words('portuguese')
print("aquele" in stopwords_list)

True


In [30]:
for i in gabarito['palavra'].to_list():
    if i.lower() in stopwords_list:
        print(i)

E
DO
DAS
DE
DA
DOS


In [31]:
gabarito_sem_stopword = gabarito[~gabarito['palavra'].str.lower().isin(stopwords_list)]

In [32]:
for i in gabarito_sem_stopword['palavra'].to_list():
    if i.lower() in stopwords_list:
        print(i)

In [33]:
gabarito_sem_stopword

,palavra,y
0,MARIETA,1
1,LUZ,1
2,ELIANE,1
3,GUSMÃO,1
4,MAURICIO,1
...,...,...
11626,ICARAI,0
11627,PROCEDIMENTO,0
11628,AMARO,0
11629,PONTUACAO,0


In [ ]:
gabarito_sem_stopword.to_csv(PATH_GABARITO, index = False)

In [34]:
# 1. Definimos o caminho absoluto exato para evitar erro de nível (../)
caminho_final = r'C:\Users\joaol\OneDrive\Desktop\MestradoBaseMédica\ProjetoDoutorado\BASES\experimento_corretor\gabarito_corretor_base_medica.csv'

In [35]:
# 2. Pegamos o diretório pai (C:\...\BASES\experimento_corretor)
diretorio = os.path.dirname(caminho_final)

In [36]:
# 4. Salvamos o seu DataFrame 'gabarito_sem_stopword'
gabarito_sem_stopword.to_csv(caminho_final, index=False)

print(f"🚀 Gabarito salvo com sucesso em: {caminho_final}")

🚀 Gabarito salvo com sucesso em: C:\Users\joaol\OneDrive\Desktop\MestradoBaseMédica\ProjetoDoutorado\BASES\experimento_corretor\gabarito_corretor_base_medica.csv
